In [2]:
import sqlite3
from pathlib import Path

import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


def load_invoice_data(db_path=None):
    """Load engineered invoice-level features from SQLite."""
    if db_path is None:
        candidates = [
            Path("ML_project/data/inventory.db"),
            Path("../data/inventory.db"),
            Path("data/inventory.db"),
        ]
        db_path = next((p for p in candidates if p.exists()), None)
        if db_path is None:
            raise FileNotFoundError("Could not find inventory.db in expected locations.")
    else:
        db_path = Path(db_path)

    query = """
    WITH purchase_agg AS (
        SELECT
            p.PONumber,
            COUNT(DISTINCT p.Brand) AS total_brands,
            SUM(p.Quantity) AS total_item_quantity,
            SUM(p.Dollars) AS total_item_dollars,
            AVG(julianday(p.ReceivingDate) - julianday(p.PODate)) AS avg_receiving_delay
        FROM purchases p
        GROUP BY p.PONumber
    )
    SELECT
        vi.PONumber,
        vi.Quantity AS invoice_quantity,
        vi.Dollars AS invoice_dollars,
        vi.Freight AS invoice_freight,
        (julianday(vi.InvoiceDate) - julianday(vi.PODate)) AS days_po_to_invoice,
        (julianday(vi.PayDate) - julianday(vi.InvoiceDate)) AS days_to_pay,
        pa.total_brands,
        pa.total_item_quantity,
        pa.total_item_dollars,
        pa.avg_receiving_delay
    FROM vendor_invoice vi
    LEFT JOIN purchase_agg pa
        ON vi.PONumber = pa.PONumber
    """

    with sqlite3.connect(str(db_path)) as conn:
        return pd.read_sql_query(query, conn)


def create_invoice_risk_label(row):
    if abs(row["invoice_dollars"] - row["total_item_dollars"]) > 5:
        return 1
    if row["avg_receiving_delay"] > 10:
        return 1
    return 0


def prepare_dataset(df):
    df = df.copy()
    df["flag_invoice"] = df.apply(create_invoice_risk_label, axis=1)

    feature_cols = [
        "invoice_quantity",
        "invoice_dollars",
        "invoice_freight",
        "total_brands",
        "total_item_quantity",
        "days_po_to_invoice",
        "total_item_dollars",
    ]

    X = df[feature_cols]
    y = df["flag_invoice"]
    return X, y


def split_and_scale(X, y, test_size=0.2, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=y,
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    return X_train, X_test, y_train, y_test, X_train_scaled, X_test_scaled, scaler


def save_preprocessing_artifacts(scaler, output_path="models/invoice_scaler.pkl"):
    output = Path(output_path)
    output.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(scaler, output)
    return output
